In [33]:
# Confere se as bibliotecas foram instaladas, mostrando a versão de cada uma.

import pandas as pd, sklearn, xgboost
print(pd.__version__, sklearn.__version__, xgboost.__version__)

3.1.1 1.9.1 3.4.1


In [34]:
# Lê o arquivo da base Bank Marketing. Nesse arquivo, o separador de colunas deve ser por ponto e vírgula (";").

df = pd.read_csv("data/bank-full.csv", sep=";")
print(df.shape)
df.head()

(45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [35]:
# Mostra um resumo das informações da tabela.

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   age        45211 non-null  int64
 1   job        45211 non-null  str  
 2   marital    45211 non-null  str  
 3   education  45211 non-null  str  
 4   default    45211 non-null  str  
 5   balance    45211 non-null  int64
 6   housing    45211 non-null  str  
 7   loan       45211 non-null  str  
 8   contact    45211 non-null  str  
 9   day        45211 non-null  int64
 10  month      45211 non-null  str  
 11  duration   45211 non-null  int64
 12  campaign   45211 non-null  int64
 13  pdays      45211 non-null  int64
 14  previous   45211 non-null  int64
 15  poutcome   45211 non-null  str  
 16  y          45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 5.9 MB


In [36]:
# Retorna o número de linhas duplicadas e de colunas com valores "unknown".
print("Duplicados:", df.duplicated().sum())

for col in df.select_dtypes(include="str").columns:
    n = (df[col] == "unknown").sum()
    if n > 0:
        print(col, n)

Duplicados: 0
job 288
education 1857
contact 13020
poutcome 36959


In [37]:
# Hipótese: o "unknown" em poutcome (resultado da campanha anterior) não é dado perdido, e sim "cliente nunca foi contatado antes"
# Hipótese confirmada.

pd.crosstab(df["poutcome"], df["previous"] == 0)

previous,False,True
poutcome,,
failure,4901,0
other,1840,0
success,1511,0
unknown,5,36954


In [38]:
# Conta quantos clientes aceitaram (yes) ou recusaram (no), em quantidade e proporção

print(df["y"].value_counts())
print(df["y"].value_counts(normalize=True))

y
no     39922
yes     5289
Name: count, dtype: int64
y
no     0.883015
yes    0.116985
Name: proportion, dtype: float64


In [39]:
# Resumo estatístico das colunas numéricas (mínimo, máximo, média) para achar valores 'estranhos'


df.describe()

,age,balance,day,duration,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,258.163080,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,257.527812,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,319.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,63.000000,871.000000,275.000000


In [40]:
# Conta casos suspeitos nas colunas numéricas (códigos como -1, zeros e extremos)

print("pdays == -1:", (df["pdays"] == -1).sum())
print("pdays == -1 e previous > 0:", ((df["pdays"] == -1) & (df["previous"] > 0)).sum())
print("duration == 0:", (df["duration"] == 0).sum())
print("balance < 0:", (df["balance"] < 0).sum())
print("previous > 50:", (df["previous"] > 50).sum())

pdays == -1: 36954
pdays == -1 e previous > 0: 0
duration == 0: 3
balance < 0: 3766
previous > 50: 4


In [41]:
dados = df.copy()
dados["previously_contacted"] = (dados["pdays"] != -1).astype(int)
dados["pdays"] = dados["pdays"].replace(-1,0)
dados = dados.drop(columns=["duration"])
y = (dados["y"] == "yes").astype(int)
X = dados.drop(columns=["y"])
print(X.shape, y.shape)

(45211, 16) (45211,)


In [42]:
# Separa os nomes das colunas numéricas e das colunas de texto

colunas_num = ["age", "balance", "day", "campaign", "pdays", "previous", "previously_contacted"]
colunas_cat = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "poutcome"]

In [43]:
# Monta o pré-processador (números na mesma escala, texto em colunas 0/1), sem treinar ainda

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessador = ColumnTransformer([
    ("num", StandardScaler(), colunas_num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), colunas_cat),
])

In [44]:
# Só para conferir quantas colunas o X terá depois da transformação

print(preprocessador.fit_transform(X).shape)

(45211, 51)


In [46]:
# Cria as 5 partições estratificadas com semente fixa (pois o grupo usa essa mesma)

from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [48]:
# Confere o tamanho e a proporção de "yes" no teste de cada partição

for i, (idx_treino, idx_teste) in enumerate(skf.split(X, y), start=1):
    print(f"Partição {i}: treino={len(idx_treino)}, teste={len(idx_teste)}, yes no teste={y.iloc[idx_teste].mean():.3f}")

Partição 1: treino=36168, teste=9043, yes no teste=0.117
Partição 2: treino=36169, teste=9042, yes no teste=0.117
Partição 3: treino=36169, teste=9042, yes no teste=0.117
Partição 4: treino=36169, teste=9042, yes no teste=0.117
Partição 5: treino=36169, teste=9042, yes no teste=0.117


In [50]:
# Teste de ponta a ponta com um KNN simples nas 5 partições (só valida a estrutura, não são resultados finais)

from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

teste = Pipeline([("prep", preprocessador), ("modelo", KNeighborsClassifier())])
print(cross_val_score(teste, X, y, cv=skf, scoring="f1"))

[0.30183549 0.32053872 0.31078366 0.28746594 0.32671958]
